[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Everyday Requests


## What you will be able to do

Answer the requests a developer gets against two databases that already exist: a college's students,
sections and grades, and a company's invoices and payments. Turn a question in somebody's words into
SQL, with joins that bring the names back, aggregates that answer a question with "per" in it, dates
that cover the right days, and updates that change what was asked and nothing else. Check what a
report says before sending it, and hand it over as a table, a chart or a CSV file.


## The idea

### The problem

Two databases are in use, and neither of them is yours to design this week. One holds a college's
students, courses, terms, sections and enrollments; the other holds a company's customers, invoices,
invoice lines and payments. The requests arrive in the words of the person asking, and never name a
table.

"Who is in Programming I, section B?" "Ana wants her transcript, and the registrar wants her GPA."
"Which sections are nearly full? We may have to open another." "What did we invoice in March, and
what actually came in?" "Which customers still owe us, and how late are they?" "The grade for one
student was entered wrong, and it has to be corrected today." The work is turning each of these into
SQL, running it against real data, and being sure enough of the answer to send it.

The same few shapes come back every time: a join to turn an id into a name, an aggregate for a
question with "per" in it, a `LEFT JOIN` for the rows with nothing on the other side, a date range
that covers the last day, and an update inside a transaction that changes exactly the rows it should.
This notebook works through them on both databases, one request at a time.

### What a request looks like in SQL

> A **request** is a question in a person's words, and answering it means deciding four things:
> which **tables** hold the values, how they **join**, which rows the question includes, and what to
> count or total. A **join** brings a row's related rows in, and a **`LEFT JOIN`** keeps rows that
> have none. An **aggregate**, `COUNT`, `SUM`, `AVG`, `MIN` or `MAX`, answers over a group of rows,
> and **`GROUP BY`** names the group, once for every value of the "per" in the question.
> **`WHERE`** chooses the rows before grouping, **`HAVING`** chooses the groups after it. A
> **grade point average** is not an average of the grades but one weighted by credits,
> `SUM(points * credits) / SUM(credits)`, and an **aging report** groups what customers still owe
> by how long it has been overdue, as of a date the report is given.

### Why it works that way

- **A request names things people use, not tables.** "Programming I, section B" is a row in
  `courses` and a row in `sections`, and the roster is in `enrollments`, so the answer joins four
  tables to print two columns.
- **Every "per" in a question is a `GROUP BY`.** Credits per student, invoiced per month, paid per
  customer: the phrase after "per" is the group, and the rest is an aggregate over it.
- **A row with nothing on the other side disappears from an inner join.** A student who has passed
  nothing, a month with no invoices, an invoice with no payment: `LEFT JOIN` keeps them, and
  `COALESCE` turns the `NULL`s that come with them into zeros.
- **Dates are text here, and text sorts.** `'2026-03-31' < '2026-04-01'`, so a month is
  `>= '2026-03-01' AND < '2026-04-01'`, which keeps the last day whether or not the column carries a
  time. `strftime` groups by month, and `julianday` subtracts two dates into days.
- **Money is counted in whole cents.** Cents are integers, and integers add up exactly, where a
  dollars-and-cents `REAL` loses a fraction of a cent at a time until a report is off by a penny.
- **A report is run as of a date.** A report that asks the clock what day it is gives a different
  answer tomorrow and cannot be checked, so the date arrives as a parameter.
- **An update says which rows it changed.** `cursor.rowcount` after an `UPDATE` is the number of rows
  written, and 0 means the `WHERE` matched nothing, which is a bug report waiting to happen.

### Where this shows up

This is most of the work in any application with a database behind it: a student information system,
an accounting package, a warehouse, a helpdesk. The requests here are the ones such systems answer
every day, and the reporting tools people put in front of a database, Metabase among them, send SQL
of exactly this shape. In this library, the **Pandas, Deep Dive** and **Matplotlib, Deep Dive**
guides take a result like the ones here and draw it, the **FastAPI, Deep Dive** guide puts the same
queries behind an HTTP request, and the **SQLAlchemy, Deep Dive** guide writes them as Python objects
instead of SQL.

### What this notebook covers

- What is in the two databases, read from the file itself
- A section's roster, from four joined tables
- A transcript, and the credit-weighted GPA and earned credits behind it
- The students an advisor has to see, with `LEFT JOIN`, `GROUP BY` and `HAVING`
- Sections near their capacity, and a timetable clash found by joining a table to itself
- A corrected grade and a moved student, each in one transaction, each counting the rows it changed
- Invoiced and collected by month, as a table, a chart and a CSV file
- What is still owed, as a view, and an aging report as of a date
- How long customers take to pay, from two dates
- A payment applied across two invoices, oldest first
- Which shape of query answers which kind of request
- The Monday morning reports for both databases
- Eight errors, from an aggregate in the wrong clause to money kept in the wrong type

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:", autocommit=True)
conn.executescript("""
    CREATE TABLE invoices (id INTEGER PRIMARY KEY, customer TEXT, issued_on TEXT,
                           amount_cents INTEGER);
    CREATE TABLE payments (id INTEGER PRIMARY KEY, invoice_id INTEGER, amount_cents INTEGER);
    INSERT INTO invoices VALUES (1, 'Northwind Foods', '2026-03-02', 120000),
                                (2, 'Kestrel Media', '2026-03-31', 45000);
    INSERT INTO payments VALUES (1, 1, 50000);
""")

# "Who still owes us for March, and how much?"
for row in conn.execute("""
    SELECT invoices.customer, invoices.issued_on,
           printf('%.2f', (invoices.amount_cents
                           - COALESCE(SUM(payments.amount_cents), 0)) / 100.0) AS owed
    FROM invoices LEFT JOIN payments ON payments.invoice_id = invoices.id
    WHERE invoices.issued_on >= '2026-03-01' AND invoices.issued_on < '2026-04-01'
    GROUP BY invoices.id
    ORDER BY invoices.issued_on
"""):
    print(tuple(row))
conn.close()
```

```
('Northwind Foods', '2026-03-02', '700.00')
('Kestrel Media', '2026-03-31', '450.00')
```

One question, and four decisions: the two tables, the `LEFT JOIN` that keeps the unpaid invoice,
the half-open date range that keeps the 31st, and the total per invoice. Everything in this notebook
is that, on bigger tables and harder questions.


## Setup

Six imports, four small helpers, and the two databases, built from formulas so that every number
here is the same on your machine.

- `sqlite3` opens both databases
- `csv` writes a report out for a spreadsheet
- `contextmanager`, from `contextlib`, turns `transaction` into a `with` block
- `datetime` and `timedelta` lay out the invoices and payments over half a year
- `Path` names the scratch folder and the two files
- `shutil` removes the scratch folder at the start and at the end

`connect` opens a connection that enforces foreign keys and returns rows by column name.
`transaction` wraps a change in `BEGIN IMMEDIATE` and `COMMIT`. `show` prints a query's rows as a
table with its column names, which is how every answer below is read. `dollars` and `bar` turn cents
into the text a report shows. `TODAY` is the date the reports are run as of.

The school database has students, courses, terms, sections, enrollments and a `grades` table of
grade points. The ledger has customers, invoices, invoice lines, payments, and the applications that
say which payment paid which invoice.


In [1]:
import csv
import shutil
import sqlite3
from contextlib import contextmanager
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
SCHOOL = SCRATCH / "school.db"
LEDGER = SCRATCH / "ledger.db"
TODAY = "2026-04-15"          # every report runs as of a date, so that it gives the same answer tomorrow


def connect(path):
    """A connection that enforces foreign keys, returns rows by column name, and writes its own transactions."""
    conn = sqlite3.connect(path, autocommit=True)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


@contextmanager
def transaction(conn):
    """Run a with block inside BEGIN IMMEDIATE and COMMIT, or ROLLBACK if anything in it raises."""
    conn.execute("BEGIN IMMEDIATE")
    try:
        yield
        conn.execute("COMMIT")
    except BaseException:
        conn.execute("ROLLBACK")
        raise


def show(cursor):
    """Print a query's rows as a table, with the column names cursor.description gives, and say how many."""
    rows = [["" if value is None else f"{value}" for value in row] for row in cursor.fetchall()]
    names = [column[0] for column in cursor.description]
    widths = [max([len(name)] + [len(row[i]) for row in rows]) for i, name in enumerate(names)]
    print("  ".join(name.ljust(width) for name, width in zip(names, widths)))
    print("  ".join("-" * width for width in widths))
    for row in rows:
        print("  ".join(value.ljust(width) for value, width in zip(row, widths)))
    print(f"({len(rows)} rows)")


def dollars(cents):
    """Cents, as the dollars people write in an email."""
    return f"${cents / 100:,.2f}"


def bar(cents, largest, width=32):
    """A bar for a chart, as long as cents is against the largest value in the report."""
    return "#" * round(width * cents / largest) if largest else ""

STUDENT_NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
TERMS = [("2025 Spring", "2025-01-13", "2025-05-02"), ("2025 Fall", "2025-08-25", "2025-12-12"),
         ("2026 Spring", "2026-01-12", "2026-05-01")]
GRADES = [("A", 4.0, 1), ("A-", 3.7, 1), ("B+", 3.3, 1), ("B", 3.0, 1), ("B-", 2.7, 1), ("C+", 2.3, 1),
          ("C", 2.0, 1), ("C-", 1.7, 1), ("D", 1.0, 1), ("F", 0.0, 0), ("W", None, 0)]
MEETINGS = [("MWF", "08:00", "08:50"), ("MWF", "09:00", "09:50"), ("TTh", "09:30", "10:45"),
            ("MWF", "11:00", "11:50"), ("TTh", "13:00", "14:15"), ("MWF", "14:00", "14:50")]
INSTRUCTORS = ["Dr. Hale", "Dr. Osei", "Dr. Ibarra", "Dr. Novak", "Dr. Mensah", "Dr. Lindgren"]
ROOMS = ["SCI 210", "SCI 118", "MAT 004", "HUM 302", "LAB 101", "HUM 210"]

CUSTOMERS = [("Northwind Foods", 30), ("Blue Harbor Design", 30), ("Granite Systems", 45),
             ("Pine Ridge Clinic", 30), ("Aster Logistics", 60), ("Kestrel Media", 45),
             ("Lantern Books", 30), ("Vela Robotics", 60)]
CATALOG = [("Consulting, senior hour", 18500), ("Consulting, standard hour", 12500),
           ("Implementation, fixed fee", 145000), ("Support, monthly", 45000),
           ("Training, per seat", 35000), ("Hosting, monthly", 25000)]
METHODS = ["ACH", "check", "card"]

SCHOOL_SCHEMA = """
    CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,
                           program TEXT NOT NULL, started_on TEXT NOT NULL) STRICT;
    CREATE TABLE courses (id INTEGER PRIMARY KEY, code TEXT NOT NULL UNIQUE, title TEXT NOT NULL,
                          department TEXT NOT NULL, credits INTEGER NOT NULL) STRICT;
    CREATE TABLE terms (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE, starts_on TEXT NOT NULL,
                        ends_on TEXT NOT NULL) STRICT;
    CREATE TABLE grades (grade TEXT PRIMARY KEY, points REAL, earns_credit INTEGER NOT NULL) STRICT;
    CREATE TABLE sections (id INTEGER PRIMARY KEY,
                           course_id INTEGER NOT NULL REFERENCES courses (id),
                           term_id INTEGER NOT NULL REFERENCES terms (id),
                           letter TEXT NOT NULL, instructor TEXT NOT NULL, room TEXT NOT NULL,
                           meets TEXT NOT NULL, starts_at TEXT NOT NULL, ends_at TEXT NOT NULL,
                           capacity INTEGER NOT NULL, UNIQUE (course_id, term_id, letter)) STRICT;
    CREATE TABLE enrollments (id INTEGER PRIMARY KEY,
                              student_id INTEGER NOT NULL REFERENCES students (id),
                              section_id INTEGER NOT NULL REFERENCES sections (id),
                              status TEXT NOT NULL CHECK (status IN ('enrolled', 'completed', 'withdrawn')),
                              grade TEXT REFERENCES grades (grade),
                              UNIQUE (student_id, section_id)) STRICT;
    CREATE INDEX enrollments_by_section ON enrollments (section_id);
"""

LEDGER_SCHEMA = """
    CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE,
                            terms_days INTEGER NOT NULL, since TEXT NOT NULL) STRICT;
    CREATE TABLE invoices (id INTEGER PRIMARY KEY, number TEXT NOT NULL UNIQUE,
                           customer_id INTEGER NOT NULL REFERENCES customers (id),
                           issued_on TEXT NOT NULL, due_on TEXT NOT NULL,
                           status TEXT NOT NULL CHECK (status IN ('open', 'void'))) STRICT;
    CREATE TABLE invoice_lines (id INTEGER PRIMARY KEY,
                                invoice_id INTEGER NOT NULL REFERENCES invoices (id),
                                description TEXT NOT NULL, quantity INTEGER NOT NULL,
                                unit_cents INTEGER NOT NULL) STRICT;
    CREATE TABLE payments (id INTEGER PRIMARY KEY,
                           customer_id INTEGER NOT NULL REFERENCES customers (id),
                           received_at TEXT NOT NULL, amount_cents INTEGER NOT NULL,
                           method TEXT NOT NULL CHECK (method IN ('ACH', 'check', 'card'))) STRICT;
    CREATE TABLE payment_applications (payment_id INTEGER NOT NULL REFERENCES payments (id),
                                       invoice_id INTEGER NOT NULL REFERENCES invoices (id),
                                       amount_cents INTEGER NOT NULL,
                                       PRIMARY KEY (payment_id, invoice_id)) STRICT;
    CREATE INDEX invoices_by_customer ON invoices (customer_id, issued_on);
"""


def build_school(path):
    """A small college: its students, courses, terms, sections and enrollments, every value from a formula."""
    conn = connect(path)
    conn.executescript(SCHOOL_SCHEMA)
    with transaction(conn):
        conn.executemany("INSERT INTO grades (grade, points, earns_credit) VALUES (?, ?, ?)", GRADES)
        conn.executemany("INSERT INTO terms (name, starts_on, ends_on) VALUES (?, ?, ?)", TERMS)
        conn.executemany("INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)", COURSES)
        conn.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)",
                         [(name, f"{name[0]}{name.split()[1]}@college.edu".lower(),
                           PROGRAMS[i % len(PROGRAMS)], TERMS[i % 2][1]) for i, name in enumerate(STUDENT_NAMES)])
        sections = []
        for t in range(len(TERMS)):
            for c in range(len(COURSES)):
                for letter in (["A", "B"] if (c + t) % 3 == 0 else ["A"]):
                    n = c * 2 + t + (letter == "B")
                    meets, starts_at, ends_at = MEETINGS[n % len(MEETINGS)]
                    sections.append((c + 1, t + 1, letter, INSTRUCTORS[n % len(INSTRUCTORS)],
                                     ROOMS[(c + 2 * t) % len(ROOMS)], meets, starts_at, ends_at,
                                     8 + (c % 3) * 2))
        conn.executemany("""INSERT INTO sections (course_id, term_id, letter, instructor, room, meets,
                            starts_at, ends_at, capacity) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""", sections)
        by_term = {}
        for row in conn.execute("SELECT id, term_id, course_id FROM sections ORDER BY id"):
            by_term.setdefault(row["term_id"], []).append((row["id"], row["course_id"]))
        enrollments = []
        for t in range(1, len(TERMS) + 1):
            for s in range(1, len(STUDENT_NAMES) + 1):
                taken = {}
                for k in range(3 + (s + t) % 2):
                    section_id, course_id = by_term[t][(s * 5 + t * 3 + k * 7) % len(by_term[t])]
                    taken[course_id] = section_id
                for course_id, section_id in taken.items():
                    if t == len(TERMS):
                        enrollments.append((s, section_id, "enrolled", None))
                    elif (s * 3 + section_id) % 23 == 0:
                        enrollments.append((s, section_id, "withdrawn", "W"))
                    else:
                        enrollments.append((s, section_id, "completed",
                                            GRADES[(s * 7 + course_id * 5 + t * 3) % 10][0]))
        conn.executemany("INSERT INTO enrollments (student_id, section_id, status, grade) VALUES (?, ?, ?, ?)",
                         enrollments)
    return conn


def build_ledger(path):
    """Half a year of invoices and payments for eight customers, every value from a formula."""
    conn = connect(path)
    conn.executescript(LEDGER_SCHEMA)
    with transaction(conn):
        conn.executemany("INSERT INTO customers (name, terms_days, since) VALUES (?, ?, ?)",
                         [(name, days, f"202{3 + i % 3}-0{1 + i % 9}-05")
                          for i, (name, days) in enumerate(CUSTOMERS)])
        conn.execute("INSERT INTO customers (name, terms_days, since) VALUES (?, ?, ?)",
                     ("Harbor Lights Studio", 30, "2026-04-06"))     # signed this month, nothing invoiced yet
        for n in range(48):
            day = datetime(2025, 10, 1) + timedelta(days=n * 4)
            customer = n % len(CUSTOMERS) + 1
            due = day + timedelta(days=CUSTOMERS[customer - 1][1])
            invoice_id = conn.execute("""INSERT INTO invoices (number, customer_id, issued_on, due_on, status)
                                         VALUES (?, ?, ?, ?, ?) RETURNING id""",
                                      (f"INV-{1000 + n}", customer, day.strftime("%Y-%m-%d"),
                                       due.strftime("%Y-%m-%d"), "void" if n in (11, 34) else "open")).fetchone()["id"]
            for line in range(1 + (n + customer) % 3):
                description, unit_cents = CATALOG[(n + line * 2) % len(CATALOG)]
                conn.execute("""INSERT INTO invoice_lines (invoice_id, description, quantity, unit_cents)
                                VALUES (?, ?, ?, ?)""", (invoice_id, description, 1 + (n + line) % 8, unit_cents))
        totals = {row["invoice_id"]: row["amount_cents"] for row in conn.execute(
            "SELECT invoice_id, SUM(quantity * unit_cents) AS amount_cents FROM invoice_lines GROUP BY invoice_id")}
        for row in conn.execute("SELECT id, customer_id, due_on, status FROM invoices ORDER BY id").fetchall():
            n = row["id"] - 1
            if row["status"] == "void" or n % 7 == 3:                    # a void invoice, or one nobody has paid
                continue
            paid = datetime.strptime(row["due_on"], "%Y-%m-%d") + timedelta(days=(n % 11) - 4)
            if paid.strftime("%Y-%m-%d") > TODAY:
                continue
            part = totals[row["id"]] if n % 5 else totals[row["id"]] // 2       # every fifth invoice is half paid
            payment_id = conn.execute("""INSERT INTO payments (customer_id, received_at, amount_cents, method)
                                         VALUES (?, ?, ?, ?) RETURNING id""",
                                      (row["customer_id"], paid.strftime("%Y-%m-%d") + f"T{9 + n % 8:02d}:{n * 7 % 60:02d}",
                                       part, METHODS[n % 3])).fetchone()["id"]
            conn.execute("""INSERT INTO payment_applications (payment_id, invoice_id, amount_cents)
                            VALUES (?, ?, ?)""", (payment_id, row["id"], part))
    return conn


school = build_school(SCHOOL)
ledger = build_ledger(LEDGER)
print("built", SCHOOL, "and", LEDGER, "| reports run as of", TODAY)


built scratch/school.db and scratch/ledger.db | reports run as of 2026-04-15


## Worked examples

### What is in these two databases

The first question about a database you did not build is what is in it. `sqlite_schema` lists the
tables, and `pragma_table_info` the columns of one:


In [2]:
TABLES = "SELECT name FROM sqlite_schema WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
for database, conn in (("school.db", school), ("ledger.db", ledger)):
    counted = []
    for row in conn.execute(TABLES).fetchall():
        rows = conn.execute(f'SELECT COUNT(*) FROM "{row["name"]}"').fetchone()[0]
        counted.append(f"{row['name']} ({rows})")
    print(f"{database}: " + ", ".join(counted))
print()
show(school.execute("""SELECT name, type, "notnull", dflt_value FROM pragma_table_info('enrollments')"""))


school.db: courses (10), enrollments (200), grades (11), sections (40), students (24), terms (3)
ledger.db: customers (9), invoice_lines (96), invoices (48), payment_applications (31), payments (31)

name        type     notnull  dflt_value
----------  -------  -------  ----------
id          INTEGER  0                  
student_id  INTEGER  1                  
section_id  INTEGER  1                  
status      TEXT     1                  
grade       TEXT     0                  
(5 rows)


An enrollment is a student in a section, with a status and, once the term is over, a grade. The
tables it points at, `students` and `sections`, are where the names are, which is why almost every
question below starts with a join.

### "Who is in Programming I, section B, this term?"

The request names a course by its title, a section by its letter and a term by its name. None of
those is in `enrollments`, so the roster joins four tables to print two columns:


In [3]:
ROSTER = """
    SELECT students.name, students.email, enrollments.status
    FROM enrollments
    JOIN students ON students.id = enrollments.student_id
    JOIN sections ON sections.id = enrollments.section_id
    JOIN courses ON courses.id = sections.course_id
    JOIN terms ON terms.id = sections.term_id
    WHERE courses.code = ? AND sections.letter = ? AND terms.name = ?
    ORDER BY students.name
"""

show(school.execute(ROSTER, ("CSC-101", "B", "2026 Spring")))


name           email                status  
-------------  -------------------  --------
Ben Okafor     bokafor@college.edu  enrolled
Grace Lin      glin@college.edu     enrolled
Keiko Tanaka   ktanaka@college.edu  enrolled
Olivia Brandt  obrandt@college.edu  enrolled
Pavel Novak    pnovak@college.edu   enrolled
Tara Nilsen    tnilsen@college.edu  enrolled
Yara Haddad    yhaddad@college.edu  enrolled
(7 rows)


Four joins, one row for every enrollment in that section, and the `WHERE` names the course, section
and term in the words the request used. `ORDER BY students.name` is not decoration: a roster with no
order is a different list every time the planner changes its mind.

### "Ana wants her transcript"

A transcript is the same join with the student fixed and the term as a column, plus the grade points
that turn a letter into a number. `grades` is a lookup table, and the join to it is a `LEFT JOIN`,
since a course still being taken has no grade at all:


In [4]:
TRANSCRIPT = """
    SELECT terms.name AS term, courses.code, courses.title, courses.credits,
           enrollments.status, enrollments.grade, grades.points
    FROM enrollments
    JOIN students ON students.id = enrollments.student_id
    JOIN sections ON sections.id = enrollments.section_id
    JOIN courses ON courses.id = sections.course_id
    JOIN terms ON terms.id = sections.term_id
    LEFT JOIN grades ON grades.grade = enrollments.grade
    WHERE students.email = ?
    ORDER BY terms.starts_on, courses.code
"""

show(school.execute(TRANSCRIPT, ("nandersen@college.edu",)))


term         code     title                       credits  status     grade  points
-----------  -------  --------------------------  -------  ---------  -----  ------
2025 Spring  HIS-110  World History               3        completed  A-     3.7   
2025 Spring  MAT-120  Calculus I                  4        withdrawn  W            
2025 Fall    CSC-101  Programming I               3        completed  F      0.0   
2025 Fall    PSY-101  Introduction to Psychology  3        completed  F      0.0   
2025 Fall    STA-200  Statistics                  3        withdrawn  W            
2026 Spring  CHE-110  General Chemistry           4        enrolled                
2026 Spring  ENG-105  Composition                 3        enrolled                
2026 Spring  HIS-110  World History               3        enrolled                
(8 rows)


Three terms in one list: two finished, one still in progress, two withdrawals carrying a `W` and no
points, and two courses failed. The rows for the current term have no grade at all, and an inner join
to `grades` would have dropped them from the transcript without a word.

### "And her GPA, and how many credits has she earned?"

Both numbers come from the same rows, and neither is an `AVG`. A GPA weighs every grade by the
credits of its course, and credits are earned only by the grades that pass:


In [5]:
STANDING = """
    SELECT students.name,
           COUNT(*) AS graded,
           SUM(courses.credits) AS attempted,
           SUM(CASE WHEN grades.earns_credit THEN courses.credits ELSE 0 END) AS earned,
           ROUND(SUM(grades.points * courses.credits) / SUM(courses.credits), 2) AS gpa
    FROM enrollments
    JOIN students ON students.id = enrollments.student_id
    JOIN sections ON sections.id = enrollments.section_id
    JOIN courses ON courses.id = sections.course_id
    JOIN grades ON grades.grade = enrollments.grade
    WHERE students.email = ? AND grades.points IS NOT NULL
    GROUP BY students.id
"""

show(school.execute(STANDING, ("nandersen@college.edu",)))


name           graded  attempted  earned  gpa 
-------------  ------  ---------  ------  ----
Noah Andersen  3       9          3       1.23
(1 rows)


`grades.points IS NOT NULL` leaves out the two withdrawals, which belong on a transcript and in no
average. The two `F`s stay in: an `F` has points, 0.0, and pulls the average down, but its
`earns_credit` is 0, so it adds nothing to the credits earned, and nine credits attempted come to
three earned. `CASE WHEN ... THEN ... ELSE ... END` is how a condition becomes a value inside an
aggregate.

### "Which students should an advisor see before registration?"

Anyone who has earned fewer than 15 credits, says the request, and the list has to include the
students who have earned none at all. That is a `LEFT JOIN` from `students`, so that a student with
no completed enrollment still gets a row, and a `HAVING` on the total:


In [6]:
ADVISING = """
    SELECT students.name, students.program,
           COALESCE(SUM(CASE WHEN grades.earns_credit THEN courses.credits END), 0) AS credits_earned,
           ROUND(SUM(grades.points * courses.credits) / SUM(courses.credits), 2) AS gpa
    FROM students
    LEFT JOIN enrollments ON enrollments.student_id = students.id AND enrollments.status = 'completed'
    LEFT JOIN sections ON sections.id = enrollments.section_id
    LEFT JOIN courses ON courses.id = sections.course_id
    LEFT JOIN grades ON grades.grade = enrollments.grade
    GROUP BY students.id
    HAVING credits_earned < ?
    ORDER BY credits_earned, students.name
"""

show(school.execute(ADVISING, (15,)))


name           program           credits_earned  gpa 
-------------  ----------------  --------------  ----
Noah Andersen  Psychology        3               1.23
Isabel Costa   Psychology        10              1.74
Yara Haddad    Psychology        10              1.7 
Rosa Delgado   Mathematics       12              2.06
Sam Ito        Psychology        13              2.18
Chloe Martin   Mathematics       14              2.29
Daniel Kim     Psychology        14              2.21
Felix Wagner   Biology           14              2.08
Grace Lin      Computer Science  14              2.83
Hassan Ali     Mathematics       14              2.66
Tara Nilsen    History           14              2.99
(11 rows)


The condition `enrollments.status = 'completed'` is in the `ON`, not the `WHERE`. In the `WHERE` it
would throw away the students the request is most interested in, the ones with no completed
enrollment at all, since their `status` is `NULL`. `HAVING` can use a name the `SELECT` computed,
`credits_earned`, because it runs after the grouping, where `WHERE` runs before it and cannot. That
name has to be one no table in the query already uses, which is a Common error below.

### "Which sections are nearly full?"

Counting enrollments per section, against the capacity on the section, with the same `LEFT JOIN`
care: a section nobody has signed up for is exactly the one the request wants to know about:


In [7]:
FILL = """
    SELECT courses.code, sections.letter, sections.instructor, sections.room,
           sections.meets || ' ' || sections.starts_at AS meets,
           COUNT(enrollments.id) AS enrolled, sections.capacity,
           sections.capacity - COUNT(enrollments.id) AS seats
    FROM sections
    JOIN courses ON courses.id = sections.course_id
    JOIN terms ON terms.id = sections.term_id
    LEFT JOIN enrollments ON enrollments.section_id = sections.id AND enrollments.status = 'enrolled'
    WHERE terms.name = ?
    GROUP BY sections.id
    HAVING seats <= ?
    ORDER BY seats, courses.code
"""

show(school.execute(FILL, ("2026 Spring", 2)))


code     letter  instructor  room     meets      enrolled  capacity  seats
-------  ------  ----------  -------  ---------  --------  --------  -----
BIO-101  A       Dr. Ibarra  LAB 101  TTh 09:30  7         8         1    
ENG-105  A       Dr. Ibarra  LAB 101  TTh 09:30  7         8         1    
STA-200  A       Dr. Ibarra  SCI 118  TTh 09:30  7         8         1    
(3 rows)


`COUNT(enrollments.id)` counts the enrollments, and counts 0 where the `LEFT JOIN` found none, where
`COUNT(*)` would count the empty row as one. That difference is a Common error below, and the reason
to count a column from the table that may be missing rather than `*`.

### "Has the timetable put two of anyone's classes on top of each other?"

Two enrollments of the same student, in sections that meet on the same days at overlapping times.
Both rows come from `enrollments`, so the table joins to itself, and comparing the two section ids
with `>` keeps every clash once rather than twice:


In [8]:
CLASHES = """
    SELECT students.name, first_course.code AS course, second_course.code AS clashes_with,
           first_section.meets, first_section.starts_at || ' to ' || first_section.ends_at AS first_time,
           second_section.starts_at || ' to ' || second_section.ends_at AS second_time
    FROM enrollments AS first
    JOIN enrollments AS second ON second.student_id = first.student_id
                              AND second.section_id > first.section_id
    JOIN sections AS first_section ON first_section.id = first.section_id
    JOIN sections AS second_section ON second_section.id = second.section_id
    JOIN courses AS first_course ON first_course.id = first_section.course_id
    JOIN courses AS second_course ON second_course.id = second_section.course_id
    JOIN students ON students.id = first.student_id
    JOIN terms ON terms.id = first_section.term_id
    WHERE terms.name = ? AND first.status = 'enrolled' AND second.status = 'enrolled'
      AND first_section.meets = second_section.meets
      AND first_section.starts_at < second_section.ends_at
      AND second_section.starts_at < first_section.ends_at
    ORDER BY students.name
"""

show(school.execute(CLASHES, ("2026 Spring",)))


name           course   clashes_with  meets  first_time      second_time   
-------------  -------  ------------  -----  --------------  --------------
Daniel Kim     MAT-120  PSY-101       MWF    08:00 to 08:50  08:00 to 08:50
Felix Wagner   BIO-101  ENG-105       TTh    09:30 to 10:45  09:30 to 10:45
Keiko Tanaka   BIO-101  STA-200       TTh    09:30 to 10:45  09:30 to 10:45
Liam Murphy    MAT-121  STA-200       TTh    09:30 to 10:45  09:30 to 10:45
Tara Nilsen    BIO-101  STA-200       TTh    09:30 to 10:45  09:30 to 10:45
Vera Kowalski  CHE-110  HIS-110       MWF    14:00 to 14:50  14:00 to 14:50
Yara Haddad    BIO-101  STA-200       TTh    09:30 to 10:45  09:30 to 10:45
(7 rows)


Two times of day overlap when each starts before the other ends, which is the whole test, and it
works on `'09:30'` and `'10:45'` as text because a 24-hour clock written with leading zeros sorts
like the time it means. Every one of these students has to be moved, which is the next request.

### "This grade was entered wrong"

A change, not a question, and the two things a change owes the person who asked are a transaction and
a count of the rows it touched:


In [9]:
CHANGE_GRADE = """
    UPDATE enrollments SET grade = ?
    WHERE id = (SELECT enrollments.id FROM enrollments
                JOIN students ON students.id = enrollments.student_id
                JOIN sections ON sections.id = enrollments.section_id
                JOIN courses ON courses.id = sections.course_id
                JOIN terms ON terms.id = sections.term_id
                WHERE students.email = ? AND courses.code = ? AND terms.name = ?)
"""


def change_grade(conn, email, code, term, grade):
    """Correct one enrollment's grade, and return how many rows changed, which should be 1."""
    with transaction(conn):
        cursor = conn.execute(CHANGE_GRADE, (grade, email, code, term))
    return cursor.rowcount


print("before:")
show(school.execute(STANDING, ("nandersen@college.edu",)))
print("rows changed:", change_grade(school, "nandersen@college.edu", "CSC-101", "2025 Fall", "C+"))
print("after:")
show(school.execute(STANDING, ("nandersen@college.edu",)))


before:
name           graded  attempted  earned  gpa 
-------------  ------  ---------  ------  ----
Noah Andersen  3       9          3       1.23
(1 rows)
rows changed: 1
after:
name           graded  attempted  earned  gpa
-------------  ------  ---------  ------  ---
Noah Andersen  3       9          6       2.0
(1 rows)


One row changed, and both numbers moved with it: the `F` became a `C+`, so the average rose and
three more credits were earned. Nothing had to be recalculated, since the GPA is not stored anywhere:
the query works it out from the grades every time it runs. A `rowcount` of 0 would have meant the
`WHERE` matched nothing, a wrong email or a course that student never took, and the answer to the
request would have been "nothing changed" rather than "done".

### "Move this student out of the clash"

Moving an enrollment to another section of the same course is one `UPDATE`, but only if that section
has a seat. The count and the update belong in one transaction: between a check and a write in two
transactions, somebody else can take the last seat:


In [10]:
SECTION_ROOM = """
    SELECT sections.id, sections.capacity,
           (SELECT COUNT(*) FROM enrollments
            WHERE enrollments.section_id = sections.id AND enrollments.status = 'enrolled') AS enrolled
    FROM sections
    JOIN courses ON courses.id = sections.course_id
    JOIN terms ON terms.id = sections.term_id
    WHERE courses.code = ? AND terms.name = ? AND sections.letter = ?
"""


def move_student(conn, email, code, term, letter):
    """Move a student's enrollment to another section of the same course, if that section has a seat."""
    with transaction(conn):
        section = conn.execute(SECTION_ROOM, (code, term, letter)).fetchone()
        if section is None:
            raise LookupError(f"{code} has no section {letter} in {term}")
        if section["enrolled"] >= section["capacity"]:
            raise RuntimeError(f"{code} {letter} is full: {section['enrolled']} of {section['capacity']}")
        cursor = conn.execute("""
            UPDATE enrollments SET section_id = ?
            WHERE section_id IN (SELECT sections.id FROM sections
                                 JOIN courses ON courses.id = sections.course_id
                                 JOIN terms ON terms.id = sections.term_id
                                 WHERE courses.code = ? AND terms.name = ?)
              AND student_id = (SELECT id FROM students WHERE email = ?)
        """, (section["id"], code, term, email))
    return cursor.rowcount


for email, code, letter in (("vkowalski@college.edu", "CHE-110", "A"), ("fwagner@college.edu", "BIO-101", "B")):
    try:
        print(email, "moved into", code, letter, "| rows changed:",
              move_student(school, email, code, "2026 Spring", letter))
    except LookupError as error:
        print(email, "not moved:", error)
show(school.execute(CLASHES, ("2026 Spring",)))


vkowalski@college.edu moved into CHE-110 A | rows changed: 1
fwagner@college.edu not moved: BIO-101 has no section B in 2026 Spring
name          course   clashes_with  meets  first_time      second_time   
------------  -------  ------------  -----  --------------  --------------
Daniel Kim    MAT-120  PSY-101       MWF    08:00 to 08:50  08:00 to 08:50
Felix Wagner  BIO-101  ENG-105       TTh    09:30 to 10:45  09:30 to 10:45
Keiko Tanaka  BIO-101  STA-200       TTh    09:30 to 10:45  09:30 to 10:45
Liam Murphy   MAT-121  STA-200       TTh    09:30 to 10:45  09:30 to 10:45
Tara Nilsen   BIO-101  STA-200       TTh    09:30 to 10:45  09:30 to 10:45
Yara Haddad   BIO-101  STA-200       TTh    09:30 to 10:45  09:30 to 10:45
(6 rows)


Vera Kowalski moved from the Chemistry section that meets on Monday, Wednesday and Friday at 14:00
to the one that meets on Tuesday and Thursday, and the clash is gone from the list. Felix Wagner
cannot be moved at all: Biology has one section this term, so the honest answer to the registrar is
that one of those two courses has to go. The check and the update are in one transaction, so a second
request for the same seat reads the seat this one has already taken, which a Common error below shows
going wrong.

### "What did we invoice, and what came in, month by month?"

The ledger's turn. An invoice's amount is not stored: it is the sum of its lines, so the amount comes
from a join and a `SUM`, and `COUNT(DISTINCT invoices.id)` counts invoices rather than lines. The
month comes from `strftime`, and the range is half open, so that the last day of March belongs to
March:


In [11]:
INVOICED_BY_MONTH = """
    SELECT strftime('%Y-%m', invoices.issued_on) AS month,
           COUNT(DISTINCT invoices.id) AS invoices,
           SUM(invoice_lines.quantity * invoice_lines.unit_cents) AS invoiced_cents
    FROM invoices
    JOIN invoice_lines ON invoice_lines.invoice_id = invoices.id
    WHERE invoices.status = 'open' AND invoices.issued_on >= :from AND invoices.issued_on < :to
    GROUP BY month
"""
COLLECTED_BY_MONTH = """
    SELECT strftime('%Y-%m', received_at) AS month, SUM(amount_cents) AS collected_cents
    FROM payments WHERE received_at >= :from AND received_at < :to GROUP BY month
"""

period = {"from": "2025-10-01", "to": "2026-07-01"}
months = [f"{2025 + (month - 1) // 12}-{(month - 1) % 12 + 1:02d}" for month in range(10, 19)]
invoiced = {row["month"]: row for row in ledger.execute(INVOICED_BY_MONTH, period)}
collected = {row["month"]: row["collected_cents"] for row in ledger.execute(COLLECTED_BY_MONTH, period)}
largest = max(row["invoiced_cents"] for row in invoiced.values())

print(f"{'month':<9}{'invoices':>9}{'invoiced':>14}{'collected':>14}   invoiced")
for month in months:
    cents = invoiced[month]["invoiced_cents"] if month in invoiced else 0
    print(f"{month:<9}{invoiced[month]['invoices'] if month in invoiced else 0:>9}"
          f"{dollars(cents):>14}{dollars(collected.get(month, 0)):>14}   {bar(cents, largest)}")


month     invoices      invoiced     collected   invoiced
2025-10          8    $34,680.00     $1,542.50   ##############################
2025-11          7    $35,615.00    $19,820.00   ###############################
2025-12          7    $28,180.00    $19,487.50   ########################
2026-01          8    $35,805.00    $33,280.00   ###############################
2026-02          6    $21,575.00    $19,350.00   ###################
2026-03          8    $37,050.00    $18,272.50   ################################
2026-04          2     $7,505.00     $6,435.00   ######
2026-05          0         $0.00         $0.00   
2026-06          0         $0.00         $0.00   


The chart's rows come from the months the report asked for, not from the months the query returned:
May and June have no invoices, and a report that let the data decide its own rows would have drawn a
chart with two months missing and nobody the wiser. April is short because the reports are run as of
15 April.

The same rows go to the analyst as a CSV file, which is what "can I have it in a spreadsheet" means:


In [12]:
REPORT = SCRATCH / "invoiced-by-month.csv"
with REPORT.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["month", "invoices", "invoiced", "collected"])
    for month in months:
        row = invoiced.get(month)
        writer.writerow([month, row["invoices"] if row else 0,
                         f"{(row['invoiced_cents'] if row else 0) / 100:.2f}",
                         f"{collected.get(month, 0) / 100:.2f}"])

print(REPORT.read_text(encoding="utf-8"), end="")


month,invoices,invoiced,collected
2025-10,8,34680.00,1542.50
2025-11,7,35615.00,19820.00
2025-12,7,28180.00,19487.50
2026-01,8,35805.00,33280.00
2026-02,6,21575.00,19350.00
2026-03,8,37050.00,18272.50
2026-04,2,7505.00,6435.00
2026-05,0,0.00,0.00
2026-06,0,0.00,0.00


`csv.writer` quotes whatever needs quoting, and `newline=""` is what the `csv` module asks for, both
as the **Files, Paths and Formats** guide has them. The amounts go out as plain numbers, `1234.50`,
because a spreadsheet reads a number and not `$1,234.50`.

### "Who still owes us, and how late are they?"

An invoice's balance is its lines less the payments applied to it, and that calculation is asked for
so often that it belongs in a view. A view is a query with a name: it stores no rows, and every
query that reads it runs it again:


In [13]:
ledger.execute("""
    CREATE VIEW invoice_balances AS
    SELECT invoices.id AS invoice_id, invoices.number, invoices.customer_id, invoices.issued_on,
           invoices.due_on, totals.amount_cents,
           totals.amount_cents - COALESCE(applied.paid_cents, 0) AS balance_cents
    FROM invoices
    JOIN (SELECT invoice_id, SUM(quantity * unit_cents) AS amount_cents
          FROM invoice_lines GROUP BY invoice_id) AS totals ON totals.invoice_id = invoices.id
    LEFT JOIN (SELECT invoice_id, SUM(amount_cents) AS paid_cents
               FROM payment_applications GROUP BY invoice_id) AS applied ON applied.invoice_id = invoices.id
    WHERE invoices.status = 'open'
""")


OLDEST_OPEN = """
    SELECT customers.name, invoice_balances.number, invoice_balances.due_on,
           CAST(julianday(:today) - julianday(invoice_balances.due_on) AS INTEGER) AS days_overdue,
           printf('%.2f', invoice_balances.amount_cents / 100.0) AS invoiced,
           printf('%.2f', invoice_balances.balance_cents / 100.0) AS still_owed
    FROM invoice_balances
    JOIN customers ON customers.id = invoice_balances.customer_id
    WHERE invoice_balances.balance_cents > 0
    ORDER BY days_overdue DESC
    LIMIT 8
"""

show(ledger.execute(OLDEST_OPEN, {"today": TODAY}))


name                number    due_on      days_overdue  invoiced  still_owed
------------------  --------  ----------  ------------  --------  ----------
Northwind Foods     INV-1000  2025-10-31  166           3085.00   1542.50   
Pine Ridge Clinic   INV-1003  2025-11-12  154           3050.00   3050.00   
Kestrel Media       INV-1005  2025-12-05  131           5975.00   2987.50   
Granite Systems     INV-1010  2025-12-25  111           1790.00   1790.00   
Blue Harbor Design  INV-1017  2026-01-07  98            875.00    875.00    
Vela Robotics       INV-1015  2026-01-29  76            4100.00   2050.00   
Northwind Foods     INV-1024  2026-02-04  70            3085.00   3085.00   
Blue Harbor Design  INV-1025  2026-02-08  66            250.00    125.00    
(8 rows)


The two subqueries total the lines and the payments separately, one row each per invoice, and they
are joined to the invoice afterwards. Totalling both in one join would multiply the rows together and
count every line once for every payment, which is the oldest mistake in reporting and one of the
Common errors below.

An aging report is the same balances, grouped by how overdue they are as of the report's date:


In [14]:
AGING = """
    SELECT CASE WHEN julianday(:today) - julianday(due_on) <= 0 THEN 'not due yet'
                WHEN julianday(:today) - julianday(due_on) <= 30 THEN '1 to 30 days'
                WHEN julianday(:today) - julianday(due_on) <= 60 THEN '31 to 60 days'
                WHEN julianday(:today) - julianday(due_on) <= 90 THEN '61 to 90 days'
                ELSE 'over 90 days' END AS overdue,
           COUNT(*) AS invoices,
           printf('%.2f', SUM(balance_cents) / 100.0) AS still_owed
    FROM invoice_balances
    WHERE balance_cents > 0
    GROUP BY overdue
    ORDER BY MIN(due_on) DESC
"""

show(ledger.execute(AGING, {"today": TODAY}))


overdue        invoices  still_owed
-------------  --------  ----------
not due yet    8         40535.00  
1 to 30 days   5         15060.00  
31 to 60 days  2         11122.50  
61 to 90 days  3         5260.00   
over 90 days   5         10245.00  
(5 rows)


`CASE` turns a number of days into the bucket the report wants, `GROUP BY` uses the bucket by its
name, and `ORDER BY MIN(due_on) DESC` puts the buckets in order without repeating the `CASE`: the
bucket with the latest due dates is the one that is not due yet. `:today` is a named placeholder, so
the same date goes into all five comparisons from one dictionary.

### "How long do our customers take to pay?"

Two dates, subtracted. `julianday` turns a date into a number of days, so the difference is in days,
and `AVG` over the payments applied to a customer's invoices is the answer people mean by "they pay
in about a month":


In [15]:
DAYS_TO_PAY = """
    SELECT customers.name, customers.terms_days AS terms,
           COUNT(*) AS payments,
           ROUND(AVG(julianday(payments.received_at) - julianday(invoices.issued_on)), 1) AS days_to_pay,
           ROUND(AVG(julianday(payments.received_at) - julianday(invoices.due_on)), 1) AS against_due
    FROM payment_applications
    JOIN payments ON payments.id = payment_applications.payment_id
    JOIN invoices ON invoices.id = payment_applications.invoice_id
    JOIN customers ON customers.id = invoices.customer_id
    GROUP BY customers.id
    ORDER BY against_due DESC
"""

show(ledger.execute(DAYS_TO_PAY))
print()
show(ledger.execute("""
    SELECT CASE CAST(strftime('%w', received_at) AS INTEGER)
               WHEN 0 THEN 'Sunday' WHEN 1 THEN 'Monday' WHEN 2 THEN 'Tuesday' WHEN 3 THEN 'Wednesday'
               WHEN 4 THEN 'Thursday' WHEN 5 THEN 'Friday' ELSE 'Saturday' END AS weekday,
           COUNT(*) AS payments, printf('%.2f', SUM(amount_cents) / 100.0) AS received
    FROM payments GROUP BY strftime('%w', received_at) ORDER BY strftime('%w', received_at)
"""))


name                terms  payments  days_to_pay  against_due
------------------  -----  --------  -----------  -----------
Northwind Foods     30     5         32.4         2.4        
Kestrel Media       45     5         47.2         2.2        
Aster Logistics     60     4         61.6         1.6        
Pine Ridge Clinic   30     3         31.5         1.5        
Lantern Books       30     4         30.9         0.9        
Granite Systems     45     3         45.8         0.8        
Vela Robotics       60     3         60.7         0.7        
Blue Harbor Design  30     4         29.7         -0.3       
(8 rows)

weekday    payments  received
---------  --------  --------
Sunday     5         17555.00
Monday     3         10467.50
Tuesday    3         6550.00 
Wednesday  5         32917.50
Thursday   4         14905.00
Friday     5         23555.00
Saturday   6         12237.50
(7 rows)


`against_due` is the number of days after the due date, so a negative average is a customer who pays
early. The second query groups payments by the day of the week they arrived, `%w` in `strftime`,
which is the shape a "when do payments come in" chart needs. Note that it groups by `%w`, the number,
and prints the name from a `CASE`: grouping by the name would order the week alphabetically.

### "A check came in for two invoices"

The check pays the oldest open invoices first, which is a rule the database cannot know and the
application has to apply. The payment and its applications go in together, and if the money does not
fit the customer's open invoices, nothing is written at all:


In [16]:
OPEN_INVOICES = """
    SELECT invoice_id, number, balance_cents FROM invoice_balances
    WHERE customer_id = ? AND balance_cents > 0 ORDER BY due_on, invoice_id
"""


def apply_payment(conn, customer, received_at, amount_cents, method):
    """Record a payment, apply it to that customer's open invoices oldest first, and return what it paid."""
    with transaction(conn):
        customer_id = conn.execute("SELECT id FROM customers WHERE name = ?", (customer,)).fetchone()["id"]
        payment_id = conn.execute("""INSERT INTO payments (customer_id, received_at, amount_cents, method)
                                     VALUES (?, ?, ?, ?) RETURNING id""",
                                  (customer_id, received_at, amount_cents, method)).fetchone()["id"]
        left, applied = amount_cents, []
        for invoice in conn.execute(OPEN_INVOICES, (customer_id,)).fetchall():
            if left == 0:
                break
            part = min(left, invoice["balance_cents"])
            conn.execute("""INSERT INTO payment_applications (payment_id, invoice_id, amount_cents)
                            VALUES (?, ?, ?)""", (payment_id, invoice["invoice_id"], part))
            applied.append((invoice["number"], dollars(part)))
            left -= part
        if left:
            raise ValueError(f"{dollars(left)} more than {customer} owes")
    return applied


CUSTOMER_BALANCE = """
    SELECT number, due_on, printf('%.2f', balance_cents / 100.0) AS still_owed FROM invoice_balances
    WHERE customer_id = (SELECT id FROM customers WHERE name = ?) AND balance_cents > 0 ORDER BY due_on
"""

show(ledger.execute(CUSTOMER_BALANCE, ("Northwind Foods",)))
print("applied:", apply_payment(ledger, "Northwind Foods", "2026-04-14T10:35", 400000, "check"))
show(ledger.execute(CUSTOMER_BALANCE, ("Northwind Foods",)))


number    due_on      still_owed
--------  ----------  ----------
INV-1000  2025-10-31  1542.50   
INV-1024  2026-02-04  3085.00   
INV-1040  2026-04-09  2535.00   
(3 rows)
applied: [('INV-1000', '$1,542.50'), ('INV-1024', '$2,457.50')]
number    due_on      still_owed
--------  ----------  ----------
INV-1024  2026-02-04  627.50    
INV-1040  2026-04-09  2535.00   
(2 rows)


One check, two invoices: the oldest was paid off and the next took what was left. The applications
table is what makes that possible, since a payment is not a column on an invoice but a row that says
how much of which payment went to which invoice. A payment larger than the balance raises before the
`COMMIT`, so a customer who overpays gets a decision from a person, not a half-written ledger.

### Which shape answers which request

| The request sounds like | The shape | Why |
|---|---|---|
| "list the ... for ..." | `SELECT` with joins and an `ORDER BY` | the names live in the tables the ids point at, and a list with no order is not the same list twice |
| "how many, or how much, per ..." | `GROUP BY` the "per", with `COUNT`, `SUM` or `AVG` | one row for every value of the thing after "per" |
| "including the ones with none" | `LEFT JOIN`, `COALESCE`, and `COUNT(a_column)` | an inner join drops them, `COUNT(*)` counts their empty row as one |
| "only the ones with more than ..." | `HAVING` on the aggregate, `WHERE` on everything else | `WHERE` runs before the grouping, `HAVING` after it |
| "in March", "this term", "as of today" | `>= start AND < the day after the end`, with the date passed in | it keeps the last day, whether or not the column has a time, and it can be run again tomorrow |
| "weighted by ..." | `SUM(value * weight) / SUM(weight)` | `AVG` weighs every row the same, which is the GPA nobody recognizes |
| "change X to Y" | `UPDATE` in a transaction, and report `cursor.rowcount` | the count says whether the change found its row |
| "and it has to be all or nothing" | one transaction around every statement, checks included | a check in its own transaction is out of date by the time the write runs |

The default for a report is a single statement that answers the whole request, ordered, with the
date and any number the request mentions passed in as parameters.

### Monday morning

The standing requests, one function for each database, as of a date. The term report says which
sections are under pressure and which students the advisors have to see; the money report says what
was invoiced and collected last month, what is still owed, and who is furthest behind:


In [17]:
def term_report(conn, term, seats=2, credits=15):
    """The Monday requests about a term: its sections near capacity, and the students to advise."""
    print(f"== {term}: sections with {seats} seats or fewer")
    show(conn.execute(FILL, (term, seats)))
    print(f"== students with fewer than {credits} credits")
    show(conn.execute(ADVISING, (credits,)))


def money_report(conn, today):
    """The Monday requests about the ledger, as of a date: last month, what is owed, and who is behind."""
    first = f"{today[:7]}-01"
    last_month = {"from": (datetime.strptime(first, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-01"),
                  "to": first}
    invoiced = conn.execute(INVOICED_BY_MONTH, last_month).fetchone()
    collected = conn.execute(COLLECTED_BY_MONTH, last_month).fetchone()
    print(f"== {last_month['from'][:7]}: invoiced {dollars(invoiced['invoiced_cents'])}"
          f" on {invoiced['invoices']} invoices, collected {dollars(collected['collected_cents'])}")
    print(f"== still owed as of {today}")
    show(conn.execute(AGING, {"today": today}))
    print("== the customers furthest behind")
    show(conn.execute("""
        SELECT customers.name, COUNT(*) AS invoices,
               MAX(CAST(julianday(:today) - julianday(invoice_balances.due_on) AS INTEGER)) AS days_overdue,
               printf('%.2f', SUM(invoice_balances.balance_cents) / 100.0) AS still_owed
        FROM invoice_balances
        JOIN customers ON customers.id = invoice_balances.customer_id
        WHERE invoice_balances.balance_cents > 0
        GROUP BY customers.id
        HAVING days_overdue > 0
        ORDER BY days_overdue DESC
        LIMIT 5
    """, {"today": today}))


term_report(school, "2026 Spring")
money_report(ledger, TODAY)


== 2026 Spring: sections with 2 seats or fewer
code     letter  instructor  room     meets      enrolled  capacity  seats
-------  ------  ----------  -------  ---------  --------  --------  -----
BIO-101  A       Dr. Ibarra  LAB 101  TTh 09:30  7         8         1    
ENG-105  A       Dr. Ibarra  LAB 101  TTh 09:30  7         8         1    
STA-200  A       Dr. Ibarra  SCI 118  TTh 09:30  7         8         1    
(3 rows)
== students with fewer than 15 credits
name           program           credits_earned  gpa 
-------------  ----------------  --------------  ----
Noah Andersen  Psychology        6               2.0 
Isabel Costa   Psychology        10              1.74
Yara Haddad    Psychology        10              1.7 
Rosa Delgado   Mathematics       12              2.06
Sam Ito        Psychology        13              2.18
Chloe Martin   Mathematics       14              2.29
Daniel Kim     Psychology        14              2.21
Felix Wagner   Biology           14         

Every line of both reports is a query from earlier in this notebook, given the term or the date the
report is run for. That is what the week's work turns into: a handful of statements that answer the
questions people keep asking, with the parts that change passed in.

### Where each part came from

| In the two reports | What it relies on | The notebook that showed it |
|---|---|---|
| `JOIN` from `enrollments` out to the names | a row's related rows, and the ids that reach them | **Tables and Queries**, **Joins** |
| `LEFT JOIN` with a condition in the `ON` | the rows with nothing on the other side, kept | **Joins** |
| `GROUP BY` with `COUNT`, `SUM` and `AVG`, and `HAVING` | one row for every group, chosen after the grouping | **SQL Syntax** |
| `strftime`, `julianday` and half-open date ranges | months, days between two dates, and the last day of a range | **SQL Syntax**, **Adapters and Converters** |
| `?` and `:today` placeholders for every value | values that never become part of the SQL text | **Parameters** |
| `sqlite3.Row` and `cursor.description` in `show` | rows and columns by name, whatever the query selected | **Row Factories** |
| cents in `INTEGER` columns, `STRICT` tables | money that adds up exactly, and columns that refuse the wrong type | **Type Affinity** |
| `transaction` around every change, `cursor.rowcount` after it | a change that happens completely or not at all, and says what it did | **Transactions**, **autocommit and isolation_level** |
| `FOREIGN KEY`, `UNIQUE` and `CHECK` in both schemas | ids that point at rows that exist, and statuses that are one of three words | **Constraints** |
| `CREATE VIEW invoice_balances` | a query worth a name, run again by everything that reads it | **Changing a Schema** |
| `enrollments_by_section` and `invoices_by_customer` | lookups that search an index instead of scanning | **Indexes and Query Plans** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/20-everyday-requests-solutions.ipynb).

**1.** "The dean wants the grade distribution of CSC-101 in 2025 Fall." Print every grade that was
given, how many students got it, and a bar for each.


In [18]:
# your code here


**2.** "Which instructor is teaching the most students this term?" Print every instructor in 2026
Spring with their sections and their enrolled students, the busiest first.


In [19]:
# your code here


**3.** "Are any rooms double-booked?" Print the pairs of sections in the same term that share a room,
meet on the same days, and overlap in time.


In [20]:
# your code here


**4.** "What did each customer invoice us for in the first quarter of 2026?" Print the customers with
their invoice count and total, biggest first, in dollars.


In [21]:
# your code here


**5.** "Invoice INV-1012 was raised by mistake." Mark it void inside a transaction, and print what
its month's invoiced total was before and after.


In [22]:
# your code here


**6.** "Which customers are all paid up as of the report date?" Print the customers with no open
balance, and how much they have paid in all.


In [23]:
# your code here


## Common errors

### sqlite3.OperationalError: misuse of aggregate: SUM()


In [24]:
ledger.execute("""
    SELECT customers.name, SUM(invoice_balances.balance_cents) AS owed
    FROM invoice_balances JOIN customers ON customers.id = invoice_balances.customer_id
    WHERE SUM(invoice_balances.balance_cents) > 100000
    GROUP BY customers.id
""").fetchall()


OperationalError: misuse of aggregate: SUM()

"Customers who owe more than a thousand dollars" sounds like a condition, so it goes into `WHERE`,
which SQLite refuses: `WHERE` chooses rows before there are any groups to total. The condition is
about the group, so it belongs in `HAVING`, which runs after the grouping and can use the name the
`SELECT` gave it:


In [25]:
show(ledger.execute("""
    SELECT customers.name, printf('%.2f', SUM(invoice_balances.balance_cents) / 100.0) AS owed
    FROM invoice_balances JOIN customers ON customers.id = invoice_balances.customer_id
    GROUP BY customers.id
    HAVING SUM(invoice_balances.balance_cents) > 100000
    ORDER BY SUM(invoice_balances.balance_cents) DESC
"""))


name                owed    
------------------  --------
Aster Logistics     26100.00
Lantern Books       21977.50
Vela Robotics       9275.00 
Pine Ridge Clinic   7800.00 
Kestrel Media       5687.50 
Northwind Foods     3162.50 
Granite Systems     2345.00 
Blue Harbor Design  1875.00 
(8 rows)


### sqlite3.IntegrityError: FOREIGN KEY constraint failed


In [26]:
with transaction(school):
    school.execute("INSERT INTO enrollments (student_id, section_id, status) VALUES (?, ?, 'enrolled')",
                   (1, 9999))


IntegrityError: FOREIGN KEY constraint failed

Section 9999 came from a spreadsheet and is not in the database, so the enrollment has nowhere to
point. The foreign key caught it, because `connect` runs `PRAGMA foreign_keys = ON`, and without that
SQLite would have written the row and left a roster nobody can print. Look the section up by what the
request actually names, and let the lookup fail with a message a person can act on:


In [27]:
def enrol(conn, email, code, term, letter):
    """Add a student to a section named by its course, term and letter, and return the new enrollment's id."""
    with transaction(conn):
        section = conn.execute(SECTION_ROOM, (code, term, letter)).fetchone()
        if section is None:
            raise LookupError(f"{code} has no section {letter} in {term}")
        return conn.execute("""
            INSERT INTO enrollments (student_id, section_id, status)
            VALUES ((SELECT id FROM students WHERE email = ?), ?, 'enrolled') RETURNING id
        """, (email, section["id"])).fetchone()["id"]


print("enrollment:", enrol(school, "areyes@college.edu", "CHE-110", "2026 Spring", "A"))


enrollment: 201


### sqlite3.IntegrityError: UNIQUE constraint failed: enrollments.student_id, enrollments.section_id


In [28]:
enrol(school, "areyes@college.edu", "CHE-110", "2026 Spring", "A")


IntegrityError: UNIQUE constraint failed: enrollments.student_id, enrollments.section_id

The registration page was submitted twice, and the second attempt broke the `UNIQUE` on the student
and the section, which is the constraint doing its job: nobody is in one section twice. A request
that may arrive twice is written so that the second time changes nothing and says so:


In [29]:
def enrol_once(conn, email, code, term, letter):
    """Add a student to a section unless that enrollment is already there, and say which happened."""
    with transaction(conn):
        section = conn.execute(SECTION_ROOM, (code, term, letter)).fetchone()
        row = conn.execute("""
            INSERT INTO enrollments (student_id, section_id, status)
            VALUES ((SELECT id FROM students WHERE email = ?), ?, 'enrolled')
            ON CONFLICT (student_id, section_id) DO NOTHING RETURNING id
        """, (email, section["id"])).fetchone()
    return "added" if row else "already enrolled"


print(enrol_once(school, "areyes@college.edu", "CHE-110", "2026 Spring", "A"))
print(enrol_once(school, "cmartin@college.edu", "CHE-110", "2026 Spring", "A"))


already enrolled
added


### No error, and every student in the list: a HAVING that found a column, not the alias


In [30]:
show(school.execute("""
    SELECT students.name,
           COALESCE(SUM(CASE WHEN grades.earns_credit THEN courses.credits END), 0) AS credits
    FROM students
    LEFT JOIN enrollments ON enrollments.student_id = students.id AND enrollments.status = 'completed'
    LEFT JOIN sections ON sections.id = enrollments.section_id
    LEFT JOIN courses ON courses.id = sections.course_id
    LEFT JOIN grades ON grades.grade = enrollments.grade
    GROUP BY students.id
    HAVING credits < 15
    ORDER BY credits DESC
    LIMIT 4
"""))


name          credits
------------  -------
Keiko Tanaka  21     
Quinn Harper  21     
Umar Farouk   21     
Liam Murphy   18     
(4 rows)


The advising list again, with the computed column called `credits`, and the students at the top of it
have 21 credits rather than fewer than 15. `courses.credits` is a column of a table this query joins,
so `credits` in the `HAVING` is that column and not the total the `SELECT` computed, and no course is
worth 15 credits, so the condition is true for every group. SQLite raised nothing, because there is
nothing wrong with the query it was given. Give a computed column a name no table in the query has,
as `ADVISING` calls it `credits_earned`:


In [31]:
show(school.execute(ADVISING, (15,)))


name           program           credits_earned  gpa 
-------------  ----------------  --------------  ----
Noah Andersen  Psychology        6               2.0 
Isabel Costa   Psychology        10              1.74
Yara Haddad    Psychology        10              1.7 
Rosa Delgado   Mathematics       12              2.06
Sam Ito        Psychology        13              2.18
Chloe Martin   Mathematics       14              2.29
Daniel Kim     Psychology        14              2.21
Felix Wagner   Biology           14              2.08
Grace Lin      Computer Science  14              2.83
Hassan Ali     Mathematics       14              2.66
Tara Nilsen    History           14              2.99
(11 rows)


### No error, and a GPA nobody recognizes: an average of the grade points


In [32]:
show(school.execute("""
    SELECT students.name,
           ROUND(AVG(grades.points), 2) AS gpa_by_avg,
           ROUND(SUM(grades.points * courses.credits) / SUM(courses.credits), 2) AS gpa_weighted
    FROM enrollments
    JOIN students ON students.id = enrollments.student_id
    JOIN sections ON sections.id = enrollments.section_id
    JOIN courses ON courses.id = sections.course_id
    JOIN grades ON grades.grade = enrollments.grade
    WHERE grades.points IS NOT NULL
    GROUP BY students.id
    HAVING gpa_by_avg != gpa_weighted
    ORDER BY students.name
    LIMIT 5
"""))


name           gpa_by_avg  gpa_weighted
-------------  ----------  ------------
Ben Okafor     2.72        2.77        
Chloe Martin   2.2         2.29        
Daniel Kim     2.22        2.21        
Elena Petrova  2.88        2.84        
Felix Wagner   2.15        2.08        
(5 rows)


`AVG(points)` weighs a 3-credit course the same as a 4-credit one, so it gives a different number
from the one on a transcript for almost every student. The rule is the same wherever a "weighted
average" is asked for: `SUM(value * weight) / SUM(weight)`, and `AVG` only when every row weighs the
same.

### No error, and a student with no courses counted as having one: COUNT(*) over a LEFT JOIN


In [33]:
with transaction(school):                                    # a student admitted this week
    school.execute("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)",
                   ("Zoe Iversen", "ziversen@college.edu", "Biology", "2026-01-12"))

show(school.execute("""
    SELECT students.name, COUNT(*) AS by_star, COUNT(enrollments.id) AS by_column
    FROM students
    LEFT JOIN enrollments ON enrollments.student_id = students.id AND enrollments.status = 'completed'
    GROUP BY students.id
    HAVING by_star != by_column
"""))


name         by_star  by_column
-----------  -------  ---------
Zoe Iversen  1        0        
(1 rows)


`COUNT(*)` counts rows, and a `LEFT JOIN` that found nothing still produces one row, with `NULL`s in
the columns of the missing table. `COUNT(enrollments.id)` counts the rows where that column is not
`NULL`, which is 0 for a student who has completed nothing. Count a column from the table that may be
missing, and the same care goes for `SUM`, which is `NULL` over no rows rather than 0, so a report
wraps it in `COALESCE`.

### No error, and a section over its capacity: the seats counted outside the transaction


In [34]:
seats = school.execute(SECTION_ROOM, ("ENG-105", "2026 Spring", "A")).fetchone()
print("the check sees", seats["capacity"] - seats["enrolled"], "seat free in ENG-105 A")
for email in ("mpatel@college.edu", "icosta@college.edu"):        # two requests, both past the same check
    if seats["enrolled"] < seats["capacity"]:
        print(email, "->", enrol_once(school, email, "ENG-105", "2026 Spring", "A"))
show(school.execute(SECTION_ROOM, ("ENG-105", "2026 Spring", "A")))


the check sees 1 seat free in ENG-105 A
mpatel@college.edu -> added
icosta@college.edu -> added
id  capacity  enrolled
--  --------  --------
36  8         9       
(1 rows)


The section had one seat, both requests read that same free seat, and both wrote, so the section now
holds one student more than it has room for. Nothing raised, because no constraint says a section
cannot go over its capacity: the rule is the application's. Read the seats inside the transaction
that writes the enrollment, as `move_student` does, so that the second request reads the seat the
first has already taken:


In [35]:
def enrol_if_room(conn, email, code, term, letter):
    """Add a student to a section, with the seats read inside the transaction that writes the enrollment."""
    with transaction(conn):
        section = conn.execute(SECTION_ROOM, (code, term, letter)).fetchone()
        if section["enrolled"] >= section["capacity"]:
            raise RuntimeError(f"{code} {letter} is full: {section['enrolled']} of {section['capacity']}")
        conn.execute("""
            INSERT INTO enrollments (student_id, section_id, status)
            VALUES ((SELECT id FROM students WHERE email = ?), ?, 'enrolled')
        """, (email, section["id"]))


for email in ("qharper@college.edu", "rdelgado@college.edu"):
    try:
        enrol_if_room(school, email, "BIO-101", "2026 Spring", "A")
        print(email, "-> added")
    except RuntimeError as error:
        print(email, "-> refused:", error)
show(school.execute(SECTION_ROOM, ("BIO-101", "2026 Spring", "A")))


qharper@college.edu -> added
rdelgado@college.edu -> refused: BIO-101 A is full: 8 of 8
id  capacity  enrolled
--  --------  --------
28  8         8       
(1 rows)


### No error, and dollars that do not add up: money kept as a REAL


In [36]:
loose = connect(SCRATCH / "loose.db")
loose.execute("CREATE TABLE lines (item TEXT NOT NULL, dollars REAL NOT NULL, cents INTEGER NOT NULL) STRICT")
with transaction(loose):
    loose.executemany("INSERT INTO lines (item, dollars, cents) VALUES (?, ?, ?)",
                      [("Support, monthly", 8.70, 870), ("Training, per seat", 0.29, 29),
                       ("Hosting, monthly", 1.15, 115)])

show(loose.execute("""
    SELECT item, printf('%.2f', dollars) AS looks_like,
           CAST(dollars * 100 AS INTEGER) AS cents_from_dollars, cents AS cents_stored
    FROM lines
"""))
print("a tenth and two tenths make three tenths:", loose.execute("SELECT 0.1 + 0.2 = 0.3").fetchone()[0])
loose.close()


item                looks_like  cents_from_dollars  cents_stored
------------------  ----------  ------------------  ------------
Support, monthly    8.70        869                 870         
Training, per seat  0.29        28                  29          
Hosting, monthly    1.15        114                 115         
(3 rows)
a tenth and two tenths make three tenths: 0


Every one of those amounts is stored as the nearest number a binary fraction can reach, and 8.70 is a
shade under 8.7, so multiplying by 100 and taking the whole part gives 869 cents rather than 870. The
error is small, and it turns up wherever money is compared, added over thousands of rows or converted
back to cents, which is why the last line is false. Keep money in whole cents in an `INTEGER` column,
where every total is exact, and divide by 100 only where the number is printed.

Last, this cell closes both connections and removes the scratch folder, with the two databases and
the report in it:


In [37]:
school.close()
ledger.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A request names things people use; the SQL names tables, so almost every answer starts by joining
  from the table that holds the rows out to the tables that hold the names.
- Every "per" in a question is a `GROUP BY`, `WHERE` chooses rows before the grouping and `HAVING`
  chooses groups after it, and an aggregate in `WHERE` is an error.
- `LEFT JOIN` keeps the rows with nothing on the other side, with the condition in the `ON`, and then
  `COUNT(a_column)` and `COALESCE(SUM(...), 0)` count and total them as zero.
- A weighted average, a GPA among them, is `SUM(value * weight) / SUM(weight)`; `AVG` weighs every
  row the same.
- Dates as `YYYY-MM-DD` text sort and compare as dates, a range is `>= start AND < the day after`,
  `strftime` groups by month or weekday, and `julianday` subtracts two dates into days.
- Money is whole cents in `INTEGER` columns, divided by 100 only when it is printed.
- A change goes in one transaction with the checks it depends on, and `cursor.rowcount` reports how
  many rows it really changed.
- A report is run as of a date that is passed in, fills the rows the request asked for, and goes out
  as a table, a chart or a CSV file.


## What is next

That is the end of this guide. You can open a SQLite database from Python and query it, join its
tables, pass every value through a placeholder, read rows by column name, control what types go in
and come out, decide where every transaction begins and ends, let constraints refuse bad data, change
a schema that already holds data, load rows in bulk, find them with indexes and full-text search, let
readers and a writer share one file, copy that file safely, and answer the requests people bring you
against a database somebody else designed.

One part of sqlite3 this guide leaves out is functions written in Python: `create_function` and
`create_aggregate` let SQL call Python, and `REGEXP`, an operator SQLite names but leaves every
application to define, is the usual first example. Changing a schema by hand, as this guide does, is
the work Alembic takes over in the **SQLAlchemy, Deep Dive** guide.

The **asyncpg and psycopg3, Deep Dive** guide comes next. It moves from a database in one file to
PostgreSQL, a server that many programs write to at once, and from sqlite3 to two drivers for it, one
asynchronous and one not.


---

&#8592; **Previous:** [A Searchable Archive](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/19-a-searchable-archive.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
